# Building an agentic content pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xkpacx/AI4TM/blob/main/week_7/agentic_content_pipeline.ipynb)

**Time**: ~40-50 minutes, most of it waiting on model calls. **Cost**: a few cents at most on Gemini's cheapest models -- this notebook makes on the order of 25-35 small calls total across embedding, labeling, and running the agent. If you haven't run [Session 1's setup guide](../week_2/lesson08_setup_guide.ipynb) yet, do that first: this notebook assumes a working Gemini API key stored in Colab Secrets (or a local `.env`).

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../week_2/lesson08_setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

This week's overview named the shift directly: last week the model answered from structured internal knowledge; this week the input is external, what people actually typed into a search engine, and the output is a decision about what to publish next. The clustering and mapping notebook earlier this week turned a search performance export into a ranked table of content gaps, each one a real question the site currently leaves unanswered. This notebook picks up from that table and builds an **agent**: a model given tools and a loop, in the definition this course has used since Week 2, that reads the table, decides what to look at, and produces a content recommendation grounded in what it actually found.

**What you'll do:**
1. Rebuild a compact version of this week's gap table, so this notebook runs on its own.
2. Define a small set of tools -- plain functions the agent can call -- over that table and the site's existing pages.
3. Build the agent's loop by hand: send the model the task, let it call tools, feed the results back, and repeat until it either has enough or hits a hard cap on how many calls it's allowed to make.
4. Run it on the highest-demand gap and read the actual transcript it produced, not one assumed in advance.
5. Name the point in this pipeline where a human has to read the output before anything reaches a page.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab: installs from an explicit list, not week_7/requirements.txt --
    # "Open in Colab" only loads this one file, so there's no repo alongside it
    # to read that file from. This list is that file's contents written out
    # directly; keep the two in sync if you add or remove a package.
    %pip install -q google-genai pandas numpy matplotlib seaborn scikit-learn
else:
    # Local: installs from week_7/requirements.txt. Looked up by directory,
    # not just filename, since the repo also has its own top-level
    # requirements.txt for whole-repo setup -- a same-named but wrong file a
    # plain filename search could grab by mistake from the repo root.
    import pathlib
    _cwd = pathlib.Path.cwd()
    if _cwd.name == "week_7" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "week_7" / "requirements.txt").exists():
        _req_path = "week_7/requirements.txt"
    elif (_cwd.parent / "week_7" / "requirements.txt").exists():
        _req_path = "../week_7/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print("Couldn't find week_7/requirements.txt from the current working directory:", _cwd)
        print("Set up your local environment first -- see Session 1's setup guide for the uv commands.")
    else:
        %pip install -q -r {_req_path}

### A note on privacy

Same rule as every other session: work only with public example data, synthetic data, or the anonymized examples this course provides. The search data this notebook runs on is entirely invented -- a fictional outdoor-apparel retailer, **Larkspur Trail Co.**, with a fictional set of pages and a fictional (but realistically shaped) search performance export -- so there's nothing here that needs protecting. When you point this same pipeline at your own site's real Search Console export later, remember that unpaid or free-quota API usage may be reviewed by humans or used to improve models; billed usage on a Gemini "Paid tier" project is not, but confirm that's actually the tier your key is on at [aistudio.google.com](https://aistudio.google.com) rather than assuming it.

In [ ]:
from google import genai
from google.colab import userdata

# Config: change this one line to point the whole notebook at a different Gemini model.
MODEL_NAME = "gemini-3.5-flash-lite"
EMBEDDING_MODEL_NAME = "gemini-embedding-001"

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))


def call_llm(prompt: str, model: str = None) -> str:
    response = client.models.generate_content(model=model or MODEL_NAME, contents=prompt)
    return response.text


def get_embeddings(texts: list) -> list:
    """Batched call to Gemini's embedding model. Returns one vector per input text, in the
    same order -- one call for the whole list is far cheaper than one call per text."""
    result = client.models.embed_content(model=EMBEDDING_MODEL_NAME, contents=texts)
    return [e.values for e in result.embeddings]


print("Connected. Model:", MODEL_NAME)

## Rebuilding this week's gap table

This week's earlier notebook walked through clustering a search performance export into questions and scoring each one against the site's existing pages, one careful step at a time -- the clustering method, the choice of `min_cluster_size`, and the three-step procedure for setting a defensible similarity threshold. The cells below rebuild a compact version of that same pipeline, condensed and lightly commented, so this notebook produces its own `gap_table` and can run start to finish on its own. If a step here looks unfamiliar, that earlier notebook is where it's explained in full.

**The data**: a synthetic Search Console-style export for Larkspur Trail Co. -- 38 query/page rows in [`data/search_console/performance_by_query.csv`](data/search_console/performance_by_query.csv), matched against 8 existing pages in [`data/search_console/site_pages.csv`](data/search_console/site_pages.csv). Both are invented for this exercise, shaped like real exports (the same `Query`, `Page`, `Clicks`, `Impressions`, `CTR`, `Position` columns Search Console itself uses) but small enough to embed and cluster in seconds.

In [ ]:
import pandas as pd
import pathlib

DATA_DIR_CANDIDATES = [
    pathlib.Path("data/search_console"),
    pathlib.Path("week_7/data/search_console"),
    pathlib.Path("../week_7/data/search_console"),
]
RAW_BASE = "https://raw.githubusercontent.com/xkpacx/AI4TM/main/week_7/data/search_console"


def load_csv(filename: str) -> pd.DataFrame:
    for d in DATA_DIR_CANDIDATES:
        path = d / filename
        if path.exists():
            return pd.read_csv(path)
    # Opened straight from the "Open in Colab" badge, this notebook has no sibling repo
    # files on disk -- fall back to reading the same file straight from GitHub.
    return pd.read_csv(f"{RAW_BASE}/{filename}")


perf_df = load_csv("performance_by_query.csv")
pages_df = load_csv("site_pages.csv")
print(f"{len(perf_df)} query/page rows, {perf_df['Query'].nunique()} distinct queries, {len(pages_df)} existing pages.")
perf_df.head()

### Clustering the queries

Each query gets embedded, then [`sklearn.cluster.HDBSCAN`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html) groups the embeddings into dense regions -- the density-based approach this week's overview described, chosen over k-means specifically so a real one-off query gets labeled as noise instead of dragged into the nearest real cluster. HDBSCAN's own default distance metric is euclidean, not cosine, so the vectors are L2-normalized first: euclidean distance between two normalized vectors is a monotonic function of their cosine similarity, which is the standard way to get a euclidean-native method to effectively cluster by cosine similarity.

`min_cluster_size` is the parameter that actually decides the output, as the overview explained -- so the cell below tries a few values and prints what each one produces before picking one, rather than guessing a single value up front.

In [ ]:
import numpy as np
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import normalize

queries = perf_df["Query"].tolist()
query_vectors = np.array(get_embeddings(queries))
query_vectors_normalized = normalize(query_vectors)

for min_cluster_size in (2, 3, 4):
    labels = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean").fit_predict(query_vectors_normalized)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    print(f"min_cluster_size={min_cluster_size}: {n_clusters} clusters, {n_noise} of {len(labels)} queries left as noise")

In [ ]:
# On a property this small, min_cluster_size=2 is the value that keeps genuine 3-7-query
# questions from fragmenting into pieces -- a real property with thousands of rows would
# need this set much higher. That's a judgment call read from the printout above, not a
# default; change it and re-read the clusters below if your own run suggests otherwise.
MIN_CLUSTER_SIZE = 2

cluster_ids = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, metric="euclidean").fit_predict(query_vectors_normalized)
perf_df["cluster_id"] = cluster_ids
perf_df["cluster_id"].value_counts().sort_index()

### Turning clusters into questions

Every cluster (everything except the `-1` noise label) becomes one row: its member queries, the demand behind them, and a description of its shape. **Query shape** here is a simplified version of the method [Pew's browsing study](https://www.pewresearch.org/short-reads/2025/07/22/google-users-are-less-likely-to-click-on-links-when-an-ai-summary-appears-in-the-results/) used to associate a query with an AI Overview appearing -- mean word count, and the share of queries that open with a question word. Pew's own criteria also weighed sentence length and part-of-speech; this is enough to flag the pattern, not to reproduce their exact percentages.

A cluster's **label** (the question stated in words) and its **intent** both come from the model, reading the member queries against a written taxonomy -- both this week's newer taxonomy (knowledge-seeking, guidance-seeking, output-seeking) and Broder's older navigational/transactional categories, since the overview was explicit that both stay useful depending on the job. A model-assigned intent label is a high-recall candidate, per the SIGIR 2025 comparison the overview cited -- so any cluster the model can't settle on one category for is flagged `ambiguous` and printed here instead of being guessed at without comment, the case a reviewer should see first.

In [ ]:
QUESTION_WORDS = {"how", "what", "when", "where", "why", "which", "who", "whose", "can", "do", "does", "is", "are"}

def mean_word_count(qs: list) -> float:
    return round(sum(len(q.split()) for q in qs) / len(qs), 1)

def pct_question_form(qs: list) -> float:
    return round(sum(q.strip().split()[0].lower() in QUESTION_WORDS for q in qs) / len(qs), 2)


clusters = []
for cid, group in perf_df[perf_df["cluster_id"] != -1].groupby("cluster_id"):
    cluster_queries = group["Query"].tolist()
    clusters.append({
        "cluster_id": int(cid),
        "queries": cluster_queries,
        "impressions": int(group["Impressions"].sum()),
        "clicks": int(group["Clicks"].sum()),
        "mean_word_count": mean_word_count(cluster_queries),
        "pct_question_form": pct_question_form(cluster_queries),
    })

n_noise = int((perf_df["cluster_id"] == -1).sum())
print(f"{len(clusters)} clusters kept as questions, {n_noise} queries left as noise.")

In [ ]:
INTENT_TAXONOMY = """
- knowledge-seeking: the goal is to understand or find out something
- guidance-seeking: the goal is help with a decision or a course of action
- output-seeking: the goal is an artefact the system produces, such as text, code, or a plan
- navigational: the goal is to reach a specific known page or account
- transactional: the goal is to complete a purchase or similar action
"""
ALLOWED_INTENTS = {"knowledge-seeking", "guidance-seeking", "output-seeking", "navigational", "transactional", "ambiguous"}


def label_cluster(cluster_queries: list) -> str:
    prompt = f"""These are real search queries a website received:
{chr(10).join('- ' + q for q in cluster_queries)}

State the single underlying question these queries share, as a short, natural phrase (the
kind you'd use as a page title). Answer with only that phrase, nothing else."""
    return call_llm(prompt).strip().strip('"')


def classify_intent(cluster_queries: list) -> str:
    prompt = f"""Taxonomy of search intent:
{INTENT_TAXONOMY}

Real search queries:
{chr(10).join('- ' + q for q in cluster_queries)}

Which single category from the taxonomy best describes what these people are trying to do?
Answer with exactly one of: knowledge-seeking, guidance-seeking, output-seeking, navigational,
transactional, ambiguous. Use "ambiguous" only if the queries genuinely split across categories
with no majority. Answer with only that one word, nothing else."""
    label = call_llm(prompt).strip().lower().strip(".")
    return label if label in ALLOWED_INTENTS else "ambiguous"


for c in clusters:
    c["cluster_label"] = label_cluster(c["queries"])
    c["intent"] = classify_intent(c["queries"])
    if c["intent"] == "ambiguous":
        print(f"cluster {c['cluster_id']} (\"{c['cluster_label']}\") needs a human read -- the model couldn't settle on one category.")

cluster_df = pd.DataFrame(clusters)
cluster_df[["cluster_id", "cluster_label", "intent", "impressions", "mean_word_count", "pct_question_form"]]

### Scoring clusters against the pages that exist

Each page is represented by its title, meta description, and H1 concatenated into one string, embedded with the same model used on the queries -- a hard requirement, since vectors from two different embedding models occupy different spaces and a similarity score between them means nothing. Each cluster is represented by its **centroid**, the mean of its member query vectors, and cosine similarity between the centroid and each page vector finds the best-matching page and how close that match actually is.

A gap flag, per the overview, isn't only a low similarity score -- a cluster of guidance-seeking or output-seeking queries whose best match is a single product page scores reasonably well on resemblance while still leaving a real question unanswered, since a product page and a buying guide serve two different jobs. Both signals get computed here: the score itself, and a simple `intent_mismatch` flag for exactly that pattern.

In [ ]:
page_texts = (pages_df["title"] + ". " + pages_df["meta_description"] + ". " + pages_df["h1"]).tolist()
page_vectors = np.array(get_embeddings(page_texts))


def cosine_similarity(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def cluster_centroid(cluster_queries: list) -> np.ndarray:
    idx = [queries.index(q) for q in cluster_queries]  # dataset is small enough that this is fine
    return np.mean([query_vectors[i] for i in idx], axis=0)


best_pages, best_scores, mismatches = [], [], []
for c in clusters:
    centroid = cluster_centroid(c["queries"])
    sims = [cosine_similarity(centroid, pv) for pv in page_vectors]
    best_idx = int(np.argmax(sims))
    best_pages.append(pages_df.iloc[best_idx]["url"])
    best_scores.append(round(sims[best_idx], 3))
    is_guidance_or_output = c["intent"] in {"guidance-seeking", "output-seeking"}
    matched_a_product_page = "/products/" in pages_df.iloc[best_idx]["url"]
    mismatches.append(is_guidance_or_output and matched_a_product_page)

cluster_df["best_page"] = best_pages
cluster_df["best_page_score"] = best_scores
cluster_df["intent_mismatch"] = mismatches
cluster_df.sort_values("best_page_score")[["cluster_label", "intent", "best_page", "best_page_score", "intent_mismatch"]]

### Setting the threshold

There's no defensible default here, per the overview's own three-step procedure: sample cluster/page pairs across the score range, have a human label each one adequately served or a gap, then choose the cutoff that best separates the two labels. The cell below prints every cluster's best match, sorted by score, so you can actually read each pair. The `human_labels` dict that follows starts filled with a **placeholder** rule (below the median score counts as a gap) purely so the notebook runs top to bottom before you've read anything -- replace those values with your own judgment after reading the printout, the same way the overview means a colleague reviewing this table would.

In [ ]:
scored = cluster_df.sort_values("best_page_score").reset_index(drop=True)
for row in scored.itertuples():
    print(f"cluster {row.cluster_id:>2} | score={row.best_page_score:.3f} | intent={row.intent} | best page: {row.best_page}")
    print(f"   label: {row.cluster_label}")
    print(f"   queries: {row.queries}")
    print()

In [ ]:
# Placeholder so this cell runs before you've read the printout above. True = gap,
# False = adequately served. Edit every value once you've actually read each pair.
median_score = scored["best_page_score"].median()
human_labels = {row.cluster_id: bool(row.best_page_score < median_score) for row in scored.itertuples()}

print("Placeholder labels -- edit these after reading the pairs above:")
for cid, is_gap in human_labels.items():
    print(f"  cluster {cid}: {'gap' if is_gap else 'adequately served'}")

In [ ]:
candidate_thresholds = [round(t, 2) for t in np.arange(0.5, 0.96, 0.05)]
best_threshold, best_agreement = None, -1.0
for t in candidate_thresholds:
    agreement = sum((row.best_page_score < t) == human_labels[row.cluster_id] for row in scored.itertuples()) / len(scored)
    if agreement > best_agreement:
        best_threshold, best_agreement = t, agreement

SIMILARITY_THRESHOLD = best_threshold
print(f"Chosen threshold: {SIMILARITY_THRESHOLD} (agrees with the labels above on {best_agreement:.0%} of clusters)")

That labeled sample is also a small test set, so the Week 3 evaluation template applies here too, exactly as the overview said it would -- precision and recall for this specific threshold, against this specific sample. The next notebook this week reuses that same template for a different step in the pipeline; worth trying here as well once you've replaced the placeholder labels with your own.

### The gap table

One row per question, ranked by demand, with the evidence behind every claim: which queries, how much impression volume, what intent, what page it currently maps to, how well, and whether the threshold and intent check together call it a gap.

In [ ]:
gap_table = cluster_df.copy()
gap_table["gap_flag"] = (gap_table["best_page_score"] < SIMILARITY_THRESHOLD) | gap_table["intent_mismatch"]
gap_table["threshold_used"] = SIMILARITY_THRESHOLD
gap_table = gap_table.sort_values("impressions", ascending=False).reset_index(drop=True)

gap_table[["cluster_id", "cluster_label", "impressions", "clicks", "intent", "mean_word_count",
           "pct_question_form", "best_page", "best_page_score", "intent_mismatch", "gap_flag"]]

What this table can't show, worth carrying forward from the overview directly: Google withholds queries issued by fewer than a few dozen users over two to three months, and Ahrefs measured anonymized queries at [46.77% of clicks across 22 billion clicks](https://ahrefs.com/blog/gsc-anonymized-queries/) -- so a real content gap can be a topic made entirely of rare searches that never appear as their own rows here, its clicks sitting inside a page's totals with no queries attached to explain them.

## From a table to an agent

This week's overview defined an **agent**, the same way this course has since Week 2: a model given **tools** (functions it can choose to call) and a **loop** (send the model the task, run whatever it asks for, feed the result back, repeat), running here on your own analysis instead of a generic assistant task. The gap table above is what the agent reads; the tools below are how it reads it.

Three tools, all read-only functions over data already sitting in this notebook -- no tool here writes anything or calls out to the open internet, which matters for what "grounded" can mean below: everything the agent says should trace back to one of these three calls.

In [ ]:
def list_gaps(min_impressions: int = 0) -> list:
    """Lists content gaps -- clusters of real search queries with no page that adequately
    answers them -- ranked by search demand (summed impressions), highest first.

    Args:
        min_impressions: only include gaps with at least this many summed impressions.
    """
    rows = gap_table[(gap_table["gap_flag"]) & (gap_table["impressions"] >= min_impressions)]
    rows = rows.sort_values("impressions", ascending=False)
    return rows[["cluster_id", "cluster_label", "impressions", "clicks", "intent",
                 "best_page", "best_page_score"]].to_dict("records")


def get_cluster_detail(cluster_id: int) -> dict:
    """Returns full detail for one cluster: its member queries, intent, query shape, and
    its best-matching existing page with the similarity score.

    Args:
        cluster_id: the numeric id of the cluster, as returned by list_gaps.
    """
    row = gap_table[gap_table["cluster_id"] == cluster_id]
    if row.empty:
        return {"error": f"no cluster with id {cluster_id}"}
    row = row.iloc[0]
    return {
        "cluster_id": int(row["cluster_id"]), "cluster_label": row["cluster_label"],
        "queries": row["queries"], "intent": row["intent"],
        "mean_word_count": float(row["mean_word_count"]), "pct_question_form": float(row["pct_question_form"]),
        "best_page": row["best_page"], "best_page_score": float(row["best_page_score"]),
        "gap_flag": bool(row["gap_flag"]),
    }


def get_page_content(url: str) -> dict:
    """Returns the title, meta description, and H1 on file for one existing page.

    Args:
        url: the page's URL, as returned by list_gaps or get_cluster_detail.
    """
    row = pages_df[pages_df["url"] == url]
    if row.empty:
        return {"error": f"no page on file with url {url}"}
    row = row.iloc[0]
    return {"url": url, "title": row["title"], "meta_description": row["meta_description"], "h1": row["h1"]}


print("Tools defined:", [f.__name__ for f in (list_gaps, get_cluster_detail, get_page_content)])

### Declaring the tools and building the loop by hand

The Gemini SDK can build a tool's schema automatically from a plain Python function's type hints and docstring, and run the whole call-execute-repeat cycle for you behind one function call. That convenience is exactly what this notebook needs to avoid, because the point here is to see the loop -- every tool call, in order, and the exact condition that ends it. So the schema for each tool is declared by hand below, `automatic_function_calling` stays off, and the loop is written out as ordinary Python.

In [ ]:
from google.genai import types

list_gaps_decl = types.FunctionDeclaration(
    name="list_gaps",
    description="Lists content gaps -- clusters of real search queries with no page that adequately answers them -- ranked by search demand, highest first.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "min_impressions": {"type": "integer", "description": "Only include gaps with at least this many summed impressions. Defaults to 0."},
        },
        "required": [],
    },
)

get_cluster_detail_decl = types.FunctionDeclaration(
    name="get_cluster_detail",
    description="Returns full detail for one cluster: its member queries, intent, query shape, and its best-matching existing page with the similarity score.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "cluster_id": {"type": "integer", "description": "The numeric id of the cluster, as returned by list_gaps."},
        },
        "required": ["cluster_id"],
    },
)

get_page_content_decl = types.FunctionDeclaration(
    name="get_page_content",
    description="Returns the title, meta description, and H1 on file for one existing page.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "The page's URL, as returned by list_gaps or get_cluster_detail."},
        },
        "required": ["url"],
    },
)

TOOLS_MAP = {"list_gaps": list_gaps, "get_cluster_detail": get_cluster_detail, "get_page_content": get_page_content}
TOOL_DECLARATIONS = [list_gaps_decl, get_cluster_detail_decl, get_page_content_decl]
AGENT_TOOL = types.Tool(function_declarations=TOOL_DECLARATIONS)
AGENT_CONFIG = types.GenerateContentConfig(tools=[AGENT_TOOL])

In [ ]:
MAX_TOOL_CALLS = 6  # the loop's hard stopping condition, independent of what the model wants


def run_agent(task: str, max_tool_calls: int = MAX_TOOL_CALLS) -> dict:
    """Runs the model in a manual tool-calling loop against `task`. Returns the final answer
    (or None if the loop bound was hit first) alongside a full transcript of every step."""
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=task)])]
    transcript = []

    for step in range(1, max_tool_calls + 1):
        response = client.models.generate_content(model=MODEL_NAME, contents=contents, config=AGENT_CONFIG)
        candidate_content = response.candidates[0].content
        contents.append(candidate_content)

        function_calls = [p.function_call for p in candidate_content.parts if p.function_call]
        if not function_calls:
            # Stopping condition 1: the model answered without asking for another tool.
            transcript.append({"step": step, "type": "final_answer"})
            return {"final_answer": response.text, "transcript": transcript, "stopped_by": "model decided it had enough"}

        response_parts = []
        for fc in function_calls:
            args = dict(fc.args) if fc.args else {}
            try:
                result = TOOLS_MAP[fc.name](**args)
            except Exception as e:
                result = {"error": str(e)}
            transcript.append({"step": step, "type": "tool_call", "tool": fc.name, "args": args, "result": result})
            response_parts.append(types.Part.from_function_response(name=fc.name, response={"result": result}))

        contents.append(types.Content(role="tool", parts=response_parts))

    # Stopping condition 2: the hard cap was hit before the model finished on its own.
    transcript.append({"step": max_tool_calls, "type": "loop_bound_hit"})
    return {"final_answer": None, "transcript": transcript, "stopped_by": f"hit the {max_tool_calls}-call cap before finishing"}

### Running it

One task, stated once: find the single highest-demand gap, look at it closely, check the page it currently gets matched to, and write a short recommendation grounded in what the tools actually returned.

In [ ]:
TASK_PROMPT = """You are a content strategist working from real search performance data.

Use the tools available to you to find the single highest-demand content gap, look at its
detail, and check the existing page it currently gets matched to. Then write a short content
recommendation: what should get built (a new page, or an addition to an existing page), a
working title, and 2-3 sentences on the angle it should take and why the existing page
doesn't already cover it. Ground every claim in what the tools actually returned -- if you
state a query, a score, or a page, it should be one the tools gave you, not one you invented.
"""

result = run_agent(TASK_PROMPT)

for step in result["transcript"]:
    if step["type"] == "tool_call":
        print(f"step {step['step']}: called {step['tool']}({step['args']})")
    elif step["type"] == "final_answer":
        print(f"step {step['step']}: model returned a final answer, no further tool call")
    else:
        print(f"step {step['step']}: loop bound hit")

print("\nstopped by:", result["stopped_by"])
print("\n--- final answer ---\n")
print(result["final_answer"])

### Tracing what actually happened

Read your own transcript above rather than taking this description on faith -- the point of running the loop is to see the actual sequence, not to assume it in advance. In a typical run, the model calls `list_gaps` first (there's nothing to reason about yet), then `get_cluster_detail` on whichever cluster came back highest-demand, then `get_page_content` on that cluster's best-matching page, and stops there because at that point it has a cluster, its queries, its intent, and the existing page's actual title and meta description -- enough to write a grounded recommendation without guessing. That's stopping condition 1: the model deciding, on its own, that another tool call wouldn't add anything. Stopping condition 2, the hard cap, exists for the run where that judgment is wrong -- a model that keeps calling tools without converging, or a bug in a tool that returns something the model keeps trying to clarify. Both conditions are visible in the transcript's own `stopped_by` value, which is the entire point of writing the loop out instead of letting the SDK's automatic version hide it.

### Where a human has to read this

Nothing this agent produced gets near a CMS on its own. Its tools are read-only fetches from real numbers, but the final recommendation is still a generation step, and a generation step can misstate what its own tools returned even when every fact it drew on was real -- the same groundedness concern Session 5 raised about GraphRAG answers. A reviewer's job here is narrow and checkable: does the cluster label, the score, and the page the recommendation names actually match what `get_cluster_detail` and `get_page_content` returned in the transcript above? That check takes a minute and catches the failure mode that matters most -- a confident recommendation built on a query, a score, or a page that was never actually retrieved. The next notebook this week turns this same question into something measured rather than eyeballed, running it across several generated briefs instead of reading one by hand.

### Scaling it: a memo per gap, not just one

The same loop, run once per gap instead of once total, each bounded independently -- a lightweight content-opportunity memo for every gap above a chosen size, rather than a single deep dive.

In [ ]:
top_gap_ids = gap_table[gap_table["gap_flag"]].sort_values("impressions", ascending=False)["cluster_id"].tolist()[:3]

memos = []
for cid in top_gap_ids:
    task = f"""Look at cluster_id {cid} using get_cluster_detail, and check its best-matching
page with get_page_content. Then write a 2-3 sentence content recommendation for this specific
cluster: what to build and why the existing page falls short. Ground it in what the tools
return -- name the actual queries, score, and page."""
    result = run_agent(task, max_tool_calls=4)
    memos.append({
        "cluster_id": cid,
        "stopped_by": result["stopped_by"],
        "recommendation": result["final_answer"],
    })

pd.DataFrame(memos)

## What this costs

**In model calls**, a single `run_agent` call against the highest-demand gap makes on the order of 4 calls to Gemini (one per loop iteration until the model stops), plus the roughly 20 embedding and labeling calls the compact gap table above needed once. The per-gap memo loop multiplies that by however many gaps it covers. None of this is expensive on a lightweight model -- Session 1's token cost guide covers the exact method (`count_tokens` against a per-token price) and its own note that pricing changes, so check [AI Studio](https://aistudio.google.com) rather than trusting a number written on a fixed date.

**In human work**: the threshold in the gap table above still needs a real reviewer's labels, not the placeholder rule this notebook fills in by default, and every agent output above still needs the check named in the previous section before it goes anywhere near a real page.

In [ ]:
# Rough token-count check on one agent turn, using the pattern from Session 1's token cost guide.
token_count = client.models.count_tokens(model=MODEL_NAME, contents=TASK_PROMPT)
print(f"The task prompt alone is ~{token_count.total_tokens} input tokens, before any tool results get appended.")
print("Each additional loop iteration adds roughly the size of that step's tool result back into the next call.")
print("Check current per-token pricing at https://aistudio.google.com before scaling this up.")

## What you built

A gap table rebuilt from a synthetic Search Console export, and an agent that reads it through three read-only tools in a loop written out by hand -- every call, every stopping condition, and the specific check a human still owes the output before it becomes a real page. The loop bound (`MAX_TOOL_CALLS`) and the tool set (three functions, none of them able to write anything) are both deliberately narrow choices, not limitations of the pattern itself: a production version of this same agent would likely have more tools (a competitor-content lookup, a CMS draft-save function) and a wider cap, and each addition is a new thing a reviewer has to trust or check, not just a new capability.

**Next in Session 6**: the same pipeline, aimed at one specific gap, produces a full landing page brief -- and that brief gets scored, not just read, using the same evaluation template Session 2 introduced.

**Questions?** Post in the Circle community.